In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
from datetime import datetime
SILVER_PATH = "/Volumes/workspace/damg7370/datastore/LA CRIME  DATA/silver/"

In [0]:
%skip
@dlt.table(
    name="la_crime_clean",
    comment="Cleaned LA crime Silver table"
)
def la_crime_clean():
    """
    Create the Silver (cleaned) LA crime table from the Bronze table.
    """

    # 🔹 Read from Bronze (choose one of these depending on your setup)
    # If bronze is a normal table:
    df = spark.table("workspace.bi_demo.la_crime_raw")
    # If bronze is a DLT table, you would use:
    # df = dlt.read("la_crime_raw")

    # =============================
    # CLEANING TRANSFORMATIONS
    # =============================

    # 1. Clean and parse dates
    df_cleaned = df \
        .withColumn(
            "date_reported_clean",
            to_date(col("Date_Rptd"), "yyyy MMM dd")
        ) \
        .withColumn(
            "date_occurred_clean",
            to_date(col("DATE_OCC"), "MM/dd/yyyy")
        )

    # 2. Clean and parse time (convert to proper time format)
    df_cleaned = df_cleaned \
        .withColumn(
            "time_occurred_clean",
            when(col("TIME_OCC").isNotNull(),
                 lpad(col("TIME_OCC"), 4, "0"))
            .otherwise(None)
        ) \
        .withColumn(
            "hour_occurred",
            when(col("time_occurred_clean").isNotNull(),
                 substring(col("time_occurred_clean"), 1, 2).cast("int"))
            .otherwise(None)
        ) \
        .withColumn(
            "minute_occurred",
            when(col("time_occurred_clean").isNotNull(),
                 substring(col("time_occurred_clean"), 3, 2).cast("int"))
            .otherwise(None)
        )

    # 3. Cast numeric fields
    df_cleaned = df_cleaned \
        .withColumn("dr_no_clean", col("DR_NO").cast("bigint")) \
        .withColumn("area_clean", col("AREA").cast("int")) \
        .withColumn("rpt_dist_no_clean", col("Rpt_Dist_No").cast("int")) \
        .withColumn("crime_code_clean", col("Crm_Cd").cast("int")) \
        .withColumn("premis_code_clean", col("Premis_Cd").cast("int")) \
        .withColumn("weapon_code_clean", col("Weapon_Used_Cd").cast("int"))

    # 4. Clean victim age
    df_cleaned = df_cleaned \
        .withColumn(
            "victim_age_clean",
            when(
                (col("Vict_Age").cast("int") >= 0) &
                (col("Vict_Age").cast("int") <= 120),
                col("Vict_Age").cast("int")
            ).otherwise(None)
        )

    # 5. Create age groups
    df_cleaned = df_cleaned \
        .withColumn(
            "age_group",
            when(col("victim_age_clean") < 18, "Under 18")
            .when((col("victim_age_clean") >= 18) & (col("victim_age_clean") < 25), "18-24")
            .when((col("victim_age_clean") >= 25) & (col("victim_age_clean") < 35), "25-34")
            .when((col("victim_age_clean") >= 35) & (col("victim_age_clean") < 45), "35-44")
            .when((col("victim_age_clean") >= 45) & (col("victim_age_clean") < 55), "45-54")
            .when((col("victim_age_clean") >= 55) & (col("victim_age_clean") < 65), "55-64")
            .when(col("victim_age_clean") >= 65, "65+")
            .otherwise("Unknown")
        )

    # 6. Clean victim sex
    df_cleaned = df_cleaned \
        .withColumn(
            "victim_sex_clean",
            when(col("Vict_Sex").isin("M", "F", "X"), col("Vict_Sex"))
            .when(col("Vict_Sex") == "H", "M")
            .when(col("Vict_Sex") == "-", None)
            .otherwise("Unknown")
        )

    # 7. Clean coordinates
    df_cleaned = df_cleaned \
        .withColumn(
            "latitude_clean",
            when(
                (col("`LAT`").cast("decimal(9,6)") != 0) &
                col("`LAT`").cast("decimal(9,6)").isNotNull(),
                col("`LAT`").cast("decimal(9,6)")
            ).otherwise(None)
        ) \
        .withColumn(
            "longitude_clean",
            when(
                (col("`LON`").cast("decimal(9,6)") != 0) &
                col("`LON`").cast("decimal(9,6)").isNotNull(),
                col("`LON`").cast("decimal(9,6)")
            ).otherwise(None)
        )

    # 8. Standardize status
    df_cleaned = df_cleaned \
        .withColumn("status_clean", upper(trim(col("`Status`"))))

    # 9. Create arrest flag
    df_cleaned = df_cleaned \
        .withColumn(
            "arrest_flag",
            when(col("status_clean").like("%ARREST%"), "Y").otherwise("N")
        )

    # 10. Determine arrest type
    df_cleaned = df_cleaned \
        .withColumn(
            "arrest_type",
            when(col("status_clean") == "INVEST CONT", "Investigation Continuing")
            .when(col("status_clean").like("%JUV%"), "Juvenile")
            .when(col("status_clean").like("%ADULT%"), "Adult")
            .when(col("status_clean").like("%OTHER%"), "Other")
            .otherwise(None)
        )

    # 11. Parse Part 1-2 crimes
    df_cleaned = df_cleaned.withColumn(
        "crime_part_clean",
        when(col("`Part_1-2`") == "1", "Part 1")
        .when(col("`Part_1-2`") == "2", "Part 2")
        .otherwise("Unknown")
    )

    # 12. Create crime category based on crime code description
    df_cleaned = df_cleaned \
        .withColumn(
            "crime_category",
            when(col("Crm_Cd_Desc").like("%THEFT%"), "Theft")
            .when(col("Crm_Cd_Desc").like("%ASSAULT%"), "Assault")
            .when(col("Crm_Cd_Desc").like("%BURGLARY%"), "Burglary")
            .when(col("Crm_Cd_Desc").like("%ROBBERY%"), "Robbery")
            .when(col("Crm_Cd_Desc").like("%VANDALISM%"), "Vandalism")
            .when(col("Crm_Cd_Desc").like("%RAPE%"), "Sexual Assault")
            .when(
                col("Crm_Cd_Desc").like("%HOMICIDE%") |
                col("Crm_Cd_Desc").like("%MURDER%"),
                "Homicide"
            )
            .when(col("Crm_Cd_Desc").like("%VEHICLE%"), "Vehicle-Related")
            .when(col("Crm_Cd_Desc").like("%FRAUD%"), "Fraud")
            .when(
                col("Crm_Cd_Desc").like("%DRUG%") |
                col("Crm_Cd_Desc").like("%NARCOTICS%"),
                "Drug-Related"
            )
            .otherwise("Other")
        )

    # 13. Create time period buckets
    df_cleaned = df_cleaned \
        .withColumn(
            "time_period",
            when((col("hour_occurred") >= 6) & (col("hour_occurred") < 12), "Morning")
            .when((col("hour_occurred") >= 12) & (col("hour_occurred") < 18), "Afternoon")
            .when((col("hour_occurred") >= 18) & (col("hour_occurred") < 24), "Evening")
            .when((col("hour_occurred") >= 0) & (col("hour_occurred") < 6), "Night")
            .otherwise("Unknown")
        )

    # 14. Create day of week from date
    df_cleaned = df_cleaned \
        .withColumn("day_of_week", dayofweek(col("date_occurred_clean"))) \
        .withColumn(
            "day_name",
            when(col("day_of_week") == 1, "Sunday")
            .when(col("day_of_week") == 2, "Monday")
            .when(col("day_of_week") == 3, "Tuesday")
            .when(col("day_of_week") == 4, "Wednesday")
            .when(col("day_of_week") == 5, "Thursday")
            .when(col("day_of_week") == 6, "Friday")
            .when(col("day_of_week") == 7, "Saturday")
            .otherwise(None)
        ) \
        .withColumn(
            "is_weekend",
            when(col("day_of_week").isin(1, 7), "Y").otherwise("N")
        )

    # 15. Extract date components
    df_cleaned = df_cleaned \
        .withColumn("year", year(col("date_occurred_clean"))) \
        .withColumn("month", month(col("date_occurred_clean"))) \
        .withColumn("month_name", date_format(col("date_occurred_clean"), "MMMM")) \
        .withColumn("quarter", quarter(col("date_occurred_clean"))) \
        .withColumn("week_of_year", weekofyear(col("date_occurred_clean"))) \
        .withColumn("day_of_month", dayofmonth(col("date_occurred_clean")))

    # 16. Create date keys for dimension tables
    df_cleaned = df_cleaned \
        .withColumn(
            "date_key",
            date_format(col("date_occurred_clean"), "yyyyMMdd").cast("int")
        ) \
        .withColumn(
            "time_key",
            when(col("time_occurred_clean").isNotNull(),
                 col("time_occurred_clean").cast("int"))
            .otherwise(None)
        )

    # 17. Clean location address
    df_cleaned = df_cleaned \
        .withColumn(
            "location_address_clean",
            regexp_replace(trim(col("`LOCATION`")), r"[\r\n]+", " ")
        )

    # 18. Create weapon category
    df_cleaned = df_cleaned.withColumn(
        "weapon_category",
        when(col("Weapon_Desc").isNull() | (col("Weapon_Desc") == ""), "No Weapon")
        .when(
            col("Weapon_Desc").like("%FIREARM%") |
            col("Weapon_Desc").like("%GUN%") |
            col("Weapon_Desc").like("%PISTOL%") |
            col("Weapon_Desc").like("%RIFLE%"),
            "Firearm"
        )
        .when(
            col("Weapon_Desc").like("%KNIFE%") |
            col("Weapon_Desc").like("%BLADE%"),
            "Knife/Blade"
        )
        .when(
            col("Weapon_Desc").like("%HANDS%") |
            col("Weapon_Desc").like("%FIST%") |
            col("Weapon_Desc").like("%FEET%"),
            "Body Parts"
        )
        .when(col("Weapon_Desc").like("%VEHICLE%"), "Vehicle")
        .otherwise("Other Weapon")
    )

    # 19. Victim descent description
    df_cleaned = df_cleaned.withColumn(
        "descent_description",
        when(col("Vict_Descent") == "A", "Asian")
        .when(col("Vict_Descent") == "B", "Black")
        .when(col("Vict_Descent") == "C", "Chinese")
        .when(col("Vict_Descent") == "F", "Filipino")
        .when(col("Vict_Descent") == "G", "Guamanian")
        .when(col("Vict_Descent") == "H", "Hispanic/Latino")
        .when(col("Vict_Descent") == "I", "American Indian")
        .when(col("Vict_Descent") == "J", "Japanese")
        .when(col("Vict_Descent") == "K", "Korean")
        .when(col("Vict_Descent") == "L", "Laotian")
        .when(col("Vict_Descent") == "O", "Other")
        .when(col("Vict_Descent") == "P", "Pacific Islander")
        .when(col("Vict_Descent") == "S", "Samoan")
        .when(col("Vict_Descent") == "U", "Hawaiian")
        .when(col("Vict_Descent") == "V", "Vietnamese")
        .when(col("Vict_Descent") == "W", "White")
        .when(col("Vict_Descent") == "X", "Unknown")
        .when(col("Vict_Descent") == "Z", "Asian Indian")
        .otherwise("Unknown")
    )

    # 20. Add data quality flags
    df_cleaned = df_cleaned \
        .withColumn(
            "has_valid_coordinates",
            when(
                (col("latitude_clean").isNotNull()) &
                (col("longitude_clean").isNotNull()),
                "Y"
            ).otherwise("N")
        ) \
        .withColumn(
            "has_victim_info",
            when(
                (col("victim_age_clean").isNotNull()) &
                (col("victim_sex_clean").isNotNull()),
                "Y"
            ).otherwise("N")
        ) \
        .withColumn(
            "data_quality_score",
            when(col("has_valid_coordinates") == "Y", 1).otherwise(0) +
            when(col("has_victim_info") == "Y", 1).otherwise(0) +
            when(col("Weapon_Desc").isNotNull(), 1).otherwise(0) +
            when(col("Cross_Street").isNotNull(), 1).otherwise(0)
        )

    # =============================
    # SELECT FINAL SILVER COLUMNS
    # =============================

    silver_columns = [
        # Record identifiers
        "dr_no_clean",
        "bronze_record_id",

        # Date and time columns
        "date_reported_clean",
        "date_occurred_clean",
        "time_occurred_clean",
        "hour_occurred",
        "minute_occurred",
        "time_period",
        "date_key",
        "time_key",

        # Date components
        "year",
        "month",
        "month_name",
        "quarter",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_weekend",

        # Location columns
        "area_clean",
        col("AREA_NAME").alias("area_name"),
        "rpt_dist_no_clean",
        "premis_code_clean",
        col("Premis_Desc").alias("premis_desc"),
        "location_address_clean",
        col("Cross_Street").alias("cross_street"),
        "latitude_clean",
        "longitude_clean",
        "has_valid_coordinates",

        # Crime columns
        "crime_code_clean",
        col("Crm_Cd_Desc").alias("crime_desc"),
        "crime_part_clean",
        "crime_category",
        col("Mocodes").alias("mocodes"),
        col("Crm_Cd_1").alias("crm_cd_1"),
        col("Crm_Cd_2").alias("crm_cd_2"),
        col("Crm_Cd_3").alias("crm_cd_3"),
        col("Crm_Cd_4").alias("crm_cd_4"),

        # Victim columns
        "victim_age_clean",
        "age_group",
        "victim_sex_clean",
        col("Vict_Descent").alias("victim_descent"),
        "descent_description",
        "has_victim_info",

        # Weapon columns
        "weapon_code_clean",
        col("Weapon_Desc").alias("weapon_desc"),
        "weapon_category",

        # Status columns
        "status_clean",
        col("Status_Desc").alias("status_desc"),
        "arrest_flag",
        "arrest_type",

        # Metadata
        "data_quality_score",
        current_timestamp().alias("silver_processing_timestamp"),
        col("ingestion_timestamp").alias("bronze_ingestion_timestamp")
    ]

    df_silver = df_cleaned.select(*silver_columns)
    df_silver = df_silver.dropDuplicates(["dr_no_clean"])

    # ⛔️ No .write() in DLT — just return the DataFrame
    return df_silver


In [0]:
@dlt.table(
    name="la_crime_clean",
    comment="Cleaned LA crime Silver table"
)
def la_crime_clean():
    """
    Silver (cleaned) LA crime table from Bronze table workspace.default.la_crime_raw
    """

    # Read from Bronze
    df = spark.table("workspace.bi_demo.la_crime_raw")

    # =============================
    # 1. Clean and parse dates
    # =============================
    df_cleaned = (
        df
        # Date reported: "2021 Apr 11 12:00:00 AM"
        .withColumn(
            "date_reported_clean",
            to_date(
                to_timestamp(col("Date_Rptd").cast("string"),
                             "yyyy MMM dd hh:mm:ss a")
            )
        )
        # Date occurred: "2020 Nov 07 12:00:00 AM"
        .withColumn(
            "date_occurred_clean",
            to_date(
                to_timestamp(col("DATE_OCC").cast("string"),
                             "yyyy MMM dd hh:mm:ss a")
            )
        )
    )

    # =============================
    # 2. Time parsing
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "time_occurred_clean",
            when(col("TIME_OCC").isNotNull(),
                 lpad(col("TIME_OCC").cast("string"), 4, "0"))
            .otherwise(None)
        )
        .withColumn(
            "hour_occurred",
            when(col("time_occurred_clean").isNotNull(),
                 substring(col("time_occurred_clean"), 1, 2).cast("int"))
            .otherwise(None)
        )
        .withColumn(
            "minute_occurred",
            when(col("time_occurred_clean").isNotNull(),
                 substring(col("time_occurred_clean"), 3, 2).cast("int"))
            .otherwise(None)
        )
    )

    # =============================
    # 3. Numeric casts
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn("dr_no_clean", col("DR_NO").cast("bigint"))
        .withColumn("area_clean", col("AREA").cast("int"))
        .withColumn("rpt_dist_no_clean", col("Rpt_Dist_No").cast("int"))
        .withColumn("crime_code_clean", col("Crm_Cd").cast("int"))
        .withColumn("premis_code_clean", col("Premis_Cd").cast("int"))
        .withColumn("weapon_code_clean", col("Weapon_Used_Cd").cast("int"))
    )

    # =============================
    # 4. Victim age + groups
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "victim_age_clean",
            when(
                (col("Vict_Age").cast("int") >= 0) &
                (col("Vict_Age").cast("int") <= 120),
                col("Vict_Age").cast("int")
            ).otherwise(None)
        )
        .withColumn(
            "age_group",
            when(col("victim_age_clean") < 18, "Under 18")
            .when((col("victim_age_clean") >= 18) & (col("victim_age_clean") < 25), "18-24")
            .when((col("victim_age_clean") >= 25) & (col("victim_age_clean") < 35), "25-34")
            .when((col("victim_age_clean") >= 35) & (col("victim_age_clean") < 45), "35-44")
            .when((col("victim_age_clean") >= 45) & (col("victim_age_clean") < 55), "45-54")
            .when((col("victim_age_clean") >= 55) & (col("victim_age_clean") < 65), "55-64")
            .when(col("victim_age_clean") >= 65, "65+")
            .otherwise("Unknown")
        )
    )

    # =============================
    # 5. Victim sex
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "victim_sex_clean",
            when(col("Vict_Sex").isin("M", "F", "X"), col("Vict_Sex"))
            .when(col("Vict_Sex") == "H", "M")
            .when(col("Vict_Sex") == "-", None)
            .otherwise("Unknown")
        )
    )

    # =============================
    # 6. Coordinates (LAT/LON)
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "latitude_clean",
            when(
                (col("LAT").cast("decimal(9,6)") != 0) &
                col("LAT").cast("decimal(9,6)").isNotNull(),
                col("LAT").cast("decimal(9,6)")
            ).otherwise(None)
        )
        .withColumn(
            "longitude_clean",
            when(
                (col("LON").cast("decimal(9,6)") != 0) &
                col("LON").cast("decimal(9,6)").isNotNull(),
                col("LON").cast("decimal(9,6)")
            ).otherwise(None)
        )
    )

    # =============================
    # 7. Status + arrest flags
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn("status_clean", upper(trim(col("Status"))))
        .withColumn(
            "arrest_flag",
            when(col("status_clean").like("%ARREST%"), "Y").otherwise("N")
        )
        .withColumn(
            "arrest_type",
            when(col("status_clean") == "INVEST CONT", "Investigation Continuing")
            .when(col("status_clean").like("%JUV%"), "Juvenile")
            .when(col("status_clean").like("%ADULT%"), "Adult")
            .when(col("status_clean").like("%OTHER%"), "Other")
            .otherwise(None)
        )
    )

    # =============================
    # 8. Part 1–2 + crime category
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "crime_part_clean",
            when(col("Part_1-2") == 1, "Part 1")
            .when(col("Part_1-2") == 2, "Part 2")
            .otherwise("Unknown")
        )
        .withColumn(
            "crime_category",
            when(col("Crm_Cd_Desc").like("%THEFT%"), "Theft")
            .when(col("Crm_Cd_Desc").like("%ASSAULT%"), "Assault")
            .when(col("Crm_Cd_Desc").like("%BURGLARY%"), "Burglary")
            .when(col("Crm_Cd_Desc").like("%ROBBERY%"), "Robbery")
            .when(col("Crm_Cd_Desc").like("%VANDALISM%"), "Vandalism")
            .when(col("Crm_Cd_Desc").like("%RAPE%"), "Sexual Assault")
            .when(
                col("Crm_Cd_Desc").like("%HOMICIDE%") |
                col("Crm_Cd_Desc").like("%MURDER%"),
                "Homicide"
            )
            .when(col("Crm_Cd_Desc").like("%VEHICLE%"), "Vehicle-Related")
            .when(col("Crm_Cd_Desc").like("%FRAUD%"), "Fraud")
            .when(
                col("Crm_Cd_Desc").like("%DRUG%") |
                col("Crm_Cd_Desc").like("%NARCOTICS%"),
                "Drug-Related"
            )
            .otherwise("Other")
        )
    )

    # =============================
    # 9. Time period buckets
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "time_period",
            when((col("hour_occurred") >= 6) & (col("hour_occurred") < 12), "Morning")
            .when((col("hour_occurred") >= 12) & (col("hour_occurred") < 18), "Afternoon")
            .when((col("hour_occurred") >= 18) & (col("hour_occurred") < 24), "Evening")
            .when((col("hour_occurred") >= 0) & (col("hour_occurred") < 6), "Night")
            .otherwise("Unknown")
        )
    )

    # =============================
    # 10. Day-of-week, month, etc.
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn("day_of_week", dayofweek(col("date_occurred_clean")))
        .withColumn(
            "day_name",
            when(col("day_of_week") == 1, "Sunday")
            .when(col("day_of_week") == 2, "Monday")
            .when(col("day_of_week") == 3, "Tuesday")
            .when(col("day_of_week") == 4, "Wednesday")
            .when(col("day_of_week") == 5, "Thursday")
            .when(col("day_of_week") == 6, "Friday")
            .when(col("day_of_week") == 7, "Saturday")
            .otherwise(None)
        )
        .withColumn(
            "is_weekend",
            when(col("day_of_week").isin(1, 7), "Y").otherwise("N")
        )
        .withColumn("year", year(col("date_occurred_clean")))
        .withColumn("month", month(col("date_occurred_clean")))
        .withColumn("month_name", date_format(col("date_occurred_clean"), "MMMM"))
        .withColumn("quarter", quarter(col("date_occurred_clean")))
        .withColumn("week_of_year", weekofyear(col("date_occurred_clean")))
        .withColumn("day_of_month", dayofmonth(col("date_occurred_clean")))
    )

    # =============================
    # 11. Date/time keys
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "date_key",
            when(col("date_occurred_clean").isNotNull(),
                 date_format(col("date_occurred_clean"), "yyyyMMdd").cast("int"))
        )
        .withColumn(
            "time_key",
            when(col("time_occurred_clean").isNotNull(),
                 col("time_occurred_clean").cast("int"))
            .otherwise(None)
        )
    )

    # =============================
    # 12. Location address
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "location_address_clean",
            regexp_replace(trim(col("LOCATION")), r"[\r\n]+", " ")
        )
    )

    # =============================
    # 13. Weapon category
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "weapon_category",
            when(col("Weapon_Desc").isNull() | (col("Weapon_Desc") == ""), "No Weapon")
            .when(
                col("Weapon_Desc").like("%FIREARM%") |
                col("Weapon_Desc").like("%GUN%") |
                col("Weapon_Desc").like("%PISTOL%") |
                col("Weapon_Desc").like("%RIFLE%"),
                "Firearm"
            )
            .when(
                col("Weapon_Desc").like("%KNIFE%") |
                col("Weapon_Desc").like("%BLADE%"),
                "Knife/Blade"
            )
            .when(
                col("Weapon_Desc").like("%HANDS%") |
                col("Weapon_Desc").like("%FIST%") |
                col("Weapon_Desc").like("%FEET%"),
                "Body Parts"
            )
            .when(col("Weapon_Desc").like("%VEHICLE%"), "Vehicle")
            .otherwise("Other Weapon")
        )
    )

    # =============================
    # 14. Victim descent description
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "descent_description",
            when(col("Vict_Descent") == "A", "Asian")
            .when(col("Vict_Descent") == "B", "Black")
            .when(col("Vict_Descent") == "C", "Chinese")
            .when(col("Vict_Descent") == "F", "Filipino")
            .when(col("Vict_Descent") == "G", "Guamanian")
            .when(col("Vict_Descent") == "H", "Hispanic/Latino")
            .when(col("Vict_Descent") == "I", "American Indian")
            .when(col("Vict_Descent") == "J", "Japanese")
            .when(col("Vict_Descent") == "K", "Korean")
            .when(col("Vict_Descent") == "L", "Laotian")
            .when(col("Vict_Descent") == "O", "Other")
            .when(col("Vict_Descent") == "P", "Pacific Islander")
            .when(col("Vict_Descent") == "S", "Samoan")
            .when(col("Vict_Descent") == "U", "Hawaiian")
            .when(col("Vict_Descent") == "V", "Vietnamese")
            .when(col("Vict_Descent") == "W", "White")
            .when(col("Vict_Descent") == "X", "Unknown")
            .when(col("Vict_Descent") == "Z", "Asian Indian")
            .otherwise("Unknown")
        )
    )

    # =============================
    # 15. Data quality flags
    # =============================
    df_cleaned = (
        df_cleaned
        .withColumn(
            "has_valid_coordinates",
            when(
                (col("latitude_clean").isNotNull()) &
                (col("longitude_clean").isNotNull()),
                "Y"
            ).otherwise("N")
        )
        .withColumn(
            "has_victim_info",
            when(
                (col("victim_age_clean").isNotNull()) &
                (col("victim_sex_clean").isNotNull()),
                "Y"
            ).otherwise("N")
        )
        .withColumn(
            "data_quality_score",
            when(col("has_valid_coordinates") == "Y", 1).otherwise(0) +
            when(col("has_victim_info") == "Y", 1).otherwise(0) +
            when(col("Weapon_Desc").isNotNull(), 1).otherwise(0) +
            when(col("Cross_Street").isNotNull(), 1).otherwise(0)
        )
    )

    # =============================
    # 16. Final Silver select
    # =============================
    silver_columns = [
        # Record identifiers
        "dr_no_clean",
        "bronze_record_id",

        # Date and time columns
        "date_reported_clean",
        "date_occurred_clean",
        "time_occurred_clean",
        "hour_occurred",
        "minute_occurred",
        "time_period",
        "date_key",
        "time_key",

        # Date components
        "year",
        "month",
        "month_name",
        "quarter",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "day_name",
        "is_weekend",

        # Location columns
        "area_clean",
        col("AREA_NAME").alias("area_name"),
        "rpt_dist_no_clean",
        "premis_code_clean",
        col("Premis_Desc").alias("premis_desc"),
        "location_address_clean",
        col("Cross_Street").alias("cross_street"),
        "latitude_clean",
        "longitude_clean",
        "has_valid_coordinates",

        # Crime columns
        "crime_code_clean",
        col("Crm_Cd_Desc").alias("crime_desc"),
        "crime_part_clean",
        "crime_category",
        col("Mocodes").alias("mocodes"),
        col("Crm_Cd_1").alias("crm_cd_1"),
        col("Crm_Cd_2").alias("crm_cd_2"),
        col("Crm_Cd_3").alias("crm_cd_3"),
        col("Crm_Cd_4").alias("crm_cd_4"),

        # Victim columns
        "victim_age_clean",
        "age_group",
        "victim_sex_clean",
        col("Vict_Descent").alias("victim_descent"),
        "descent_description",
        "has_victim_info",

        # Weapon columns
        "weapon_code_clean",
        col("Weapon_Desc").alias("weapon_desc"),
        "weapon_category",

        # Status columns
        "status_clean",
        col("Status_Desc").alias("status_desc"),
        "arrest_flag",
        "arrest_type",

        # Metadata
        "data_quality_score",
        current_timestamp().alias("silver_processing_timestamp"),
        col("ingestion_timestamp").alias("bronze_ingestion_timestamp"),
    ]

    df_silver = df_cleaned.select(*silver_columns).dropDuplicates(["dr_no_clean"])

    return df_silver

In [0]:
def validate_silver_layer(silver_table_name):
    """
    Validate Silver layer data quality
    """
    df_silver = (
        spark.read
        .format("delta")
        .load("/Volumes/workspace/damg7370/datastore/LA CRIME  DATA/silver/la_crime_clean")
    )
    
    print("\n" + "=" * 80)
    print("SILVER LAYER VALIDATION REPORT")
    print("=" * 80)
    
    # Record count
    total_records = df_silver.count()
    print(f"\n✅ Total records in Silver layer: {total_records:,}")
    
    # Check for duplicates
    duplicate_count = df_silver.groupBy("dr_no_clean").count().filter("count > 1").count()
    print(f"✅ Duplicate DR_NO records: {duplicate_count}")
    
    # Data quality metrics
    print("\n📊 DATA QUALITY METRICS:")
    
    quality_metrics = df_silver.agg(
        count(when(col("has_valid_coordinates") == "Y", 1)).alias("records_with_coordinates"),
        count(when(col("has_victim_info") == "Y", 1)).alias("records_with_victim_info"),
        count(when(col("arrest_flag") == "Y", 1)).alias("arrest_records"),
        count(when(col("weapon_code_clean").isNotNull(), 1)).alias("records_with_weapon"),
        avg("data_quality_score").alias("avg_quality_score"),
        min("date_occurred_clean").alias("min_date"),
        max("date_occurred_clean").alias("max_date")
    ).collect()[0]
    
    print(f"   Records with valid coordinates: {quality_metrics['records_with_coordinates']:,} ({quality_metrics['records_with_coordinates']/total_records*100:.1f}%)")
    print(f"   Records with victim info: {quality_metrics['records_with_victim_info']:,} ({quality_metrics['records_with_victim_info']/total_records*100:.1f}%)")
    print(f"   Arrest records: {quality_metrics['arrest_records']:,} ({quality_metrics['arrest_records']/total_records*100:.1f}%)")
    print(f"   Records with weapon info: {quality_metrics['records_with_weapon']:,} ({quality_metrics['records_with_weapon']/total_records*100:.1f}%)")
    print(f"   Average data quality score: {quality_metrics['avg_quality_score']:.2f}/4")
    print(f"   Date range: {quality_metrics['min_date']} to {quality_metrics['max_date']}")
    
    # Crime category distribution
    print("\n🔍 CRIME CATEGORY DISTRIBUTION:")
    crime_dist = df_silver.groupBy("crime_category") \
        .count() \
        .orderBy(desc("count")) \
        .limit(10)
    
    crime_dist.show()
    
    # Time period distribution
    print("\n⏰ TIME PERIOD DISTRIBUTION:")
    time_dist = df_silver.groupBy("time_period") \
        .count() \
        .orderBy("time_period")
    
    time_dist.show()
    
    return df_silver

# Validate Silver layer
validate_silver_layer("la_crime_clean")
